# Using Dask Local Cluster

In [1]:
import os
import xarray as xr
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pystac_client
from scipy import ndimage as ndi
from distributed import LocalCluster
from pyproj import Transformer
import json
import geopandas as gpd
from shapely.geometry import Point, box
from shapely.ops import unary_union
import pathlib
import re

cluster = LocalCluster(processes=False)
client = cluster.get_client()
cluster

def extract_time(ds):
    date_format = "%Y%m%dT%H%M%S"
    filename = ds.encoding["source"]
    date_str = os.path.basename(filename).split("_")[2]
    time = datetime.strptime(date_str, date_format)
    return ds.assign_coords(time=time)

spatial_extent = {
    "west": 3.2865,
    "south": 50.7589,
    "east": 3.7752,
    "north": 50.9842,
}

bbox_4326 = [
    spatial_extent["west"],
    spatial_extent["south"],
    spatial_extent["east"],
    spatial_extent["north"],
]

# Spatial slice parameters
x_slice = slice(549400, 554400)
y_slice = slice(5641500, 5636500)

# Connect to the STAC catalog
catalog = pystac_client.Client.open("https://stac.core.eopf.eodc.eu")

# Search for Sentinel-2 L2A items within a specific bounding box and date range
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox_4326,
    datetime="2018-11-01/2020-02-01",
)

# Retrieve the list of matching items
items = list(search.items())
hrefs = [item.assets["product"].href for item in items]

datacube = xr.open_mfdataset(
    hrefs,
    engine="zarr",
    chunks={},
    group="/measurements/reflectance/r10m",
    concat_dim="time",
    combine="nested",
    preprocess=extract_time,
    mask_and_scale=True,
).sortby("time", ascending=True).sel(x=x_slice, y=y_slice)

scl = xr.open_mfdataset(
    hrefs,
    engine="zarr",
    chunks={},
    group="/conditions/mask/l2a_classification/r20m",  # Adjust if necessary
    concat_dim="time",
    combine="nested",
    preprocess=extract_time,
    mask_and_scale=True,
).sortby("time", ascending=True)[["scl"]]


b11 = xr.open_mfdataset(
    hrefs,
    engine="zarr",
    chunks={},
    group="/measurements/reflectance/r20m",  # Adjust if necessary
    concat_dim="time",
    combine="nested",
    preprocess=extract_time,
    mask_and_scale=True,
).sortby("time", ascending=True)[["b11"]]

scl_resampled = scl.scl.interp_like(datacube, method="nearest")
b11_resampled = b11.b11.interp_like(datacube, method="nearest")

datacube["scl"] = scl_resampled
datacube["b11"] = b11_resampled

datacube = datacube.rio.write_crs("EPSG:32631")  # ensure CRS

datacube = datacube.compute()

datacube

/opt/conda/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 80.24 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


<xarray.Dataset> Size: 2GB
Dimensions:      (time: 179, y: 500, x: 500)
Coordinates:
  * x            (x) float32 2kB 5.494e+05 5.494e+05 ... 5.544e+05 5.544e+05
  * y            (y) float32 2kB 5.641e+06 5.641e+06 ... 5.637e+06 5.637e+06
  * time         (time) datetime64[ns] 1kB 2018-11-02T10:52:09 ... 2020-01-31...
    spatial_ref  int64 8B 0
Data variables:
    b02          (time, y, x) float64 358MB 0.0295 0.0272 ... 0.5136 0.504
    b03          (time, y, x) float64 358MB 0.063 0.0586 0.0608 ... 0.4632 0.456
    b04          (time, y, x) float64 358MB 0.0278 0.0263 ... 0.4228 0.4168
    b08          (time, y, x) float64 358MB 0.5619 0.506 ... 0.4452 0.4424
    scl          (time, y, x) float64 358MB 4.0 4.0 4.0 4.0 ... 9.0 9.0 9.0 9.0
    b11          (time, y, x) float64 358MB 0.1903 0.1903 ... 0.2831 0.2831

2025-10-07 15:15:49,383 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 7.10 GiB -- Worker memory limit: 10.00 GiB


In [2]:
def circular_kernel(radius: int) -> np.ndarray:
    """Create a 2D circular (disk) kernel with given radius in pixels."""
    r = int(radius)
    y, x = np.ogrid[-r : r + 1, -r : r + 1]
    k = (x * x + y * y) <= (r * r)
    return k.astype(np.float32)


def _dilate_with_convolve(mask_2d: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """
    Dilate a boolean mask using convolution with a (0/1) kernel.
    Returns a boolean array where any overlap with the kernel sets True.
    """
    if mask_2d.dtype != np.float32:
        mask_2d = mask_2d.astype(np.float32, copy=False)
    # Convolution counts how many True pixels fall under the kernel footprint.
    conv = ndi.convolve(mask_2d, kernel, mode="constant", cval=0.0)
    return conv > 0.0


def _apply_dilation_block(
    scl_block: np.ndarray, k1: np.ndarray, k2: np.ndarray
) -> np.ndarray:
    """
    Core block function (works on 2D arrays). Returns combined dilated mask (bool).
    Implements the openEO SCL logic used by mask_scl_dilation.
    """
    # S2 Sen2Cor SCL classes (for reference):
    #  0 NODATA, 1 SAT/DEF, 2 DARK, 3 CLOUD_SHADOW, 4 VEG, 5 NOT_VEG,
    #  6 WATER, 7 UNCLASS, 8 CLOUD_MED, 9 CLOUD_HIGH, 10 THIN_CIRRUS, 11 SNOW
    # openEO mask_scl_dilation uses:
    #   mask1 = everything except {2,4,5,6,7}  (aggressive neighbourhood removal)
    #   mask2 = {3,8,9,10,11}                  (clouds, shadows, snow)
    scl = scl_block.astype(np.int16, copy=False)

    mask1 = (scl != 2) & (scl != 4) & (scl != 5) & (scl != 6) & (scl != 7)
    mask2 = (scl == 3) | (scl == 8) | (scl == 9) | (scl == 10) | (scl == 11)

    dil1 = _dilate_with_convolve(mask1, kernel1)
    dil2 = _dilate_with_convolve(mask2, kernel2)

    combined = dil1 | dil2
    return combined


def mask_scl_dilation(
    ds: xr.Dataset, *, time_band_name: str = "time", scl_band_name: str = "scl"
) -> xr.Dataset:
    """
    Apply the mask_scl_dilation to an xarray Dataset containing a Sentinel-2 SCL band.

    Parameters
    ----------
    ds : xr.Dataset
        Must contain an integer SCL classification band.
        Expected dims include (y, x) and optionally time-like dims (e.g. t).
    time_band_name: str
        Name of the time dimension in 'ds'
    scl_band_name : str
        Name of the SCL band in `ds`.

    Returns
    -------
    xr.Dataset
        Same as input, with selected bands masked (set to NaN) wherever the dilated mask is True.
    """

    if scl_band_name not in ds:
        raise ValueError(
            f"{scl_band_name!r} not found in dataset variables: {list(ds.data_vars)}"
        )

    # Validate time dimension exists in dataset
    if time_band_name not in ds.dims:
        raise ValueError(
            f"{time_band_name!r} not found in dataset dimensions: {list(ds.dims)}"
        )

    scl = ds[scl_band_name]

    # Figure out which dims are spatial:
    # Try common names first, fallback to guessing the last two dims are (y, x)
    cand_xy = [d for d in ["y", "x"] if d in scl.dims]
    if len(cand_xy) != 2:
        # Guess last two dims in order
        cand_xy = list(scl.dims[-2:])
    ydim, xdim = cand_xy

    # Build dilated mask with apply_ufunc so it's Dask-friendly and vectorized over non-spatial dims.
    dilated = xr.apply_ufunc(
        _apply_dilation_block,
        scl,
        input_core_dims=[[ydim, xdim]],
        output_core_dims=[[ydim, xdim]],
        kwargs={"k1": kernel1.astype(np.float32), "k2": kernel2.astype(np.float32)},
        dask="parallelized",
        vectorize=True,
        output_dtypes=[bool],
        dask_gufunc_kwargs={"allow_rechunk": True},
    ).rename("dilated_mask")

    bands_to_mask = [v for v in ds.data_vars if v != scl_band_name]

    # Align masking array to each band (xarray broadcasting handles extra dims)
    out = ds.copy()
    for var in bands_to_mask:
        da = out[var]
        # Mask (set to NaN where dilated mask is True)
        # Preserve dtype for integer bands by promoting to float to allow NaNs
        if np.issubdtype(da.dtype, np.integer):
            da = da.astype(np.float32)
        out[var] = da.where(~dilated)

    out = out.drop_vars(scl_band_name)

    # Build a keep mask over time dimension
    keep = xr.concat(
        [out[v].notnull().any((ydim, xdim)) for v in bands_to_mask], dim="vars"
    ).any("vars")

    # Apply time filtering
    out = out.sel({time_band_name: keep})

    return out

kernel1 = circular_kernel(radius=8)
kernel2 = circular_kernel(radius=100)



In [ ]:
# Usage examples
datacube_chunked = datacube.chunk({"time": 1, "x": -1, "y": -1})  # No chunking on spatial dims
masked_ds = mask_scl_dilation(datacube, time_band_name="time", scl_band_name="scl")
masked_ds # Dask Kernel Crashes here! when i run on my local VM, it shows me: "UserWarning: Sending large graph of size 2.00 GiB.". Probably there is some config in the backend that prevents the kernl to run if the dask graph exceeds certain value!

# Using Dask Gateway

In [1]:
from dask_gateway import Gateway


# Simplest way - creates everything automatically
gate = Gateway()
cluster = gate.new_cluster()
cluster

In [ ]:
import os
import xarray as xr
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pystac_client
from scipy import ndimage as ndi
from pyproj import Transformer
import json
import geopandas as gpd
from shapely.geometry import Point, box
from shapely.ops import unary_union
import pathlib
import re


def extract_time(ds):
    date_format = "%Y%m%dT%H%M%S"
    filename = ds.encoding["source"]
    date_str = os.path.basename(filename).split("_")[2]
    time = datetime.strptime(date_str, date_format)
    return ds.assign_coords(time=time)

spatial_extent = {
    "west": 3.2865,
    "south": 50.7589,
    "east": 3.7752,
    "north": 50.9842,
}

bbox_4326 = [
    spatial_extent["west"],
    spatial_extent["south"],
    spatial_extent["east"],
    spatial_extent["north"],
]

# Spatial slice parameters
x_slice = slice(549400, 554400)
y_slice = slice(5641500, 5636500)

# Connect to the STAC catalog
catalog = pystac_client.Client.open("https://stac.core.eopf.eodc.eu")

# Search for Sentinel-2 L2A items within a specific bounding box and date range
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox_4326,
    datetime="2018-11-01/2020-02-01",
)

# Retrieve the list of matching items
items = list(search.items())
hrefs = [item.assets["product"].href for item in items]

datacube = xr.open_mfdataset(
    hrefs,
    engine="zarr",
    chunks={},
    group="/measurements/reflectance/r10m",
    concat_dim="time",
    combine="nested",
    preprocess=extract_time,
    mask_and_scale=True,
).sortby("time", ascending=True).sel(x=x_slice, y=y_slice)

scl = xr.open_mfdataset(
    hrefs,
    engine="zarr",
    chunks={},
    group="/conditions/mask/l2a_classification/r20m",  # Adjust if necessary
    concat_dim="time",
    combine="nested",
    preprocess=extract_time,
    mask_and_scale=True,
).sortby("time", ascending=True)[["scl"]]


b11 = xr.open_mfdataset(
    hrefs,
    engine="zarr",
    chunks={},
    group="/measurements/reflectance/r20m",  # Adjust if necessary
    concat_dim="time",
    combine="nested",
    preprocess=extract_time,
    mask_and_scale=True,
).sortby("time", ascending=True)[["b11"]]

scl_resampled = scl.scl.interp_like(datacube, method="nearest")
b11_resampled = b11.b11.interp_like(datacube, method="nearest")

datacube["scl"] = scl_resampled
datacube["b11"] = b11_resampled

datacube = datacube.rio.write_crs("EPSG:32631")  # ensure CRS

datacube = datacube.compute()

datacube

In [ ]:
def circular_kernel(radius: int) -> np.ndarray:
    """Create a 2D circular (disk) kernel with given radius in pixels."""
    r = int(radius)
    y, x = np.ogrid[-r : r + 1, -r : r + 1]
    k = (x * x + y * y) <= (r * r)
    return k.astype(np.float32)


def _dilate_with_convolve(mask_2d: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """
    Dilate a boolean mask using convolution with a (0/1) kernel.
    Returns a boolean array where any overlap with the kernel sets True.
    """
    if mask_2d.dtype != np.float32:
        mask_2d = mask_2d.astype(np.float32, copy=False)
    # Convolution counts how many True pixels fall under the kernel footprint.
    conv = ndi.convolve(mask_2d, kernel, mode="constant", cval=0.0)
    return conv > 0.0


def _apply_dilation_block(
    scl_block: np.ndarray, k1: np.ndarray, k2: np.ndarray
) -> np.ndarray:
    """
    Core block function (works on 2D arrays). Returns combined dilated mask (bool).
    Implements the openEO SCL logic used by mask_scl_dilation.
    """
    # S2 Sen2Cor SCL classes (for reference):
    #  0 NODATA, 1 SAT/DEF, 2 DARK, 3 CLOUD_SHADOW, 4 VEG, 5 NOT_VEG,
    #  6 WATER, 7 UNCLASS, 8 CLOUD_MED, 9 CLOUD_HIGH, 10 THIN_CIRRUS, 11 SNOW
    # openEO mask_scl_dilation uses:
    #   mask1 = everything except {2,4,5,6,7}  (aggressive neighbourhood removal)
    #   mask2 = {3,8,9,10,11}                  (clouds, shadows, snow)
    scl = scl_block.astype(np.int16, copy=False)

    mask1 = (scl != 2) & (scl != 4) & (scl != 5) & (scl != 6) & (scl != 7)
    mask2 = (scl == 3) | (scl == 8) | (scl == 9) | (scl == 10) | (scl == 11)

    dil1 = _dilate_with_convolve(mask1, kernel1)
    dil2 = _dilate_with_convolve(mask2, kernel2)

    combined = dil1 | dil2
    return combined


def mask_scl_dilation(
    ds: xr.Dataset, *, time_band_name: str = "time", scl_band_name: str = "scl"
) -> xr.Dataset:
    """
    Apply the mask_scl_dilation to an xarray Dataset containing a Sentinel-2 SCL band.

    Parameters
    ----------
    ds : xr.Dataset
        Must contain an integer SCL classification band.
        Expected dims include (y, x) and optionally time-like dims (e.g. t).
    time_band_name: str
        Name of the time dimension in 'ds'
    scl_band_name : str
        Name of the SCL band in `ds`.

    Returns
    -------
    xr.Dataset
        Same as input, with selected bands masked (set to NaN) wherever the dilated mask is True.
    """

    if scl_band_name not in ds:
        raise ValueError(
            f"{scl_band_name!r} not found in dataset variables: {list(ds.data_vars)}"
        )

    # Validate time dimension exists in dataset
    if time_band_name not in ds.dims:
        raise ValueError(
            f"{time_band_name!r} not found in dataset dimensions: {list(ds.dims)}"
        )

    scl = ds[scl_band_name]

    # Figure out which dims are spatial:
    # Try common names first, fallback to guessing the last two dims are (y, x)
    cand_xy = [d for d in ["y", "x"] if d in scl.dims]
    if len(cand_xy) != 2:
        # Guess last two dims in order
        cand_xy = list(scl.dims[-2:])
    ydim, xdim = cand_xy

    # Build dilated mask with apply_ufunc so it's Dask-friendly and vectorized over non-spatial dims.
    dilated = xr.apply_ufunc(
        _apply_dilation_block,
        scl,
        input_core_dims=[[ydim, xdim]],
        output_core_dims=[[ydim, xdim]],
        kwargs={"k1": kernel1.astype(np.float32), "k2": kernel2.astype(np.float32)},
        dask="parallelized",
        vectorize=True,
        output_dtypes=[bool],
        dask_gufunc_kwargs={"allow_rechunk": True},
    ).rename("dilated_mask")

    bands_to_mask = [v for v in ds.data_vars if v != scl_band_name]

    # Align masking array to each band (xarray broadcasting handles extra dims)
    out = ds.copy()
    for var in bands_to_mask:
        da = out[var]
        # Mask (set to NaN where dilated mask is True)
        # Preserve dtype for integer bands by promoting to float to allow NaNs
        if np.issubdtype(da.dtype, np.integer):
            da = da.astype(np.float32)
        out[var] = da.where(~dilated)

    out = out.drop_vars(scl_band_name)

    # Build a keep mask over time dimension
    keep = xr.concat(
        [out[v].notnull().any((ydim, xdim)) for v in bands_to_mask], dim="vars"
    ).any("vars")

    # Apply time filtering
    out = out.sel({time_band_name: keep})

    return out

kernel1 = circular_kernel(radius=8)
kernel2 = circular_kernel(radius=100)

In [ ]:
# Usage examples
datacube_chunked = datacube.chunk({"time": 1, "x": -1, "y": -1})  # No chunking on spatial dims
masked_ds = mask_scl_dilation(datacube, time_band_name="time", scl_band_name="scl")
masked_ds # Dask Kernel Crashes here! when i run on my local VM, it shows me: "UserWarning: Sending large graph of size 2.00 GiB.". Probably there is some config in the backend that prevents the kernl to run if the dask graph exceeds certain value!